# DoorKey: no improvement in the current experiment

**The saved three-seed experiment provides no positive evidence for our approach over RECO-greedy.**

- **Correct abstraction:** same return, same success, same supervisor use.
- **Door state ignored:** lower return and more requests for help with our approach.
- **100% success does not mean independent success:** the supervisor helps complete episodes.

These conclusions describe `doorkey_smoke.csv` (size 5, five noisy-expert trajectories per seed, three seeds). They do not establish that every possible DoorKey configuration is unhelpful.

The main comparison below contains only the two relevant methods. Other baselines and diagnostics remain available in the CSV and in the optional details at the end.


In [1]:
from pathlib import Path
import sys
import subprocess
import pandas as pd

PROJECT = next(
    (p for base in [Path.cwd(), *Path.cwd().parents]
     for p in (base, base / "research_project")
     if (p / "src" / "doorkey.py").exists()), None
)
if PROJECT is None:
    raise RuntimeError("Open this notebook within the repository")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

RESULT_FILE = PROJECT / "results" / "doorkey_smoke.csv"
results = pd.read_csv(RESULT_FILE)


## Main comparison

**Return:** higher is better; it is not a percentage.  
**Help requests / episode:** lower is better; it counts every supervisor request.  
**Episodes needing help (%):** episodes with at least one request.  
**Success (%):** episodes reaching the goal, including those helped by the supervisor.

Each table averages seeds within one size, dataset, and trajectory budget. Conditions are not pooled.


In [2]:
METHODS = ["RECO-greedy", "Ours"]
CONDITIONS = {"correct": "Correct abstraction", "ignore_door": "Door state ignored"}
LABELS = {
    "ret": "Return ↑",
    "queries_per_ep": "Help requests / episode ↓",
    "pct_ep_with_query": "Episodes needing help (%) ↓",
    "pct_success": "Success (%) ↑",
}
comparison = results[results.agent.isin(METHODS)]
for (environment, dataset, budget), frame in comparison.groupby(["env", "dataset", "n_traj"]):
    print(f"{environment} | {dataset} | {budget} trajectories | {frame.seed.nunique()} seeds")
    table = frame.groupby(["abstraction", "agent"])[list(LABELS)].mean()
    table = table.reindex(pd.MultiIndex.from_product(
        [[key for key in CONDITIONS if key in frame.abstraction.unique()], METHODS],
        names=["abstraction", "agent"],
    ))
    table = table.rename(index=CONDITIONS, level="abstraction").rename(columns=LABELS)
    table.index.names = ["Condition", "Method"]
    display(table.style.format("{:.2f}"))


doorkey-5 | noisy_expert | 5 trajectories | 3 seeds


## What changed versus RECO-greedy?

For the saved smoke experiment:

| Condition | Change in return | Change in help requests | Interpretation |
| --- | ---: | ---: | --- |
| Correct abstraction | 0.00 | 0.00 per episode | No benefit |
| Door state ignored | -5.13 | +0.30 per episode (+60%) | Worse return and more help |

With door state ignored, requests increase from **0.50 to 0.80 per episode**.
The +60% is a relative increase in the number of requests, **not** a 60-percentage-point change in success or episodes needing help.

**Conclusion: this implementation and small experiment do not support a performance advantage for our approach. Under the incorrect abstraction, our approach is worse than RECO-greedy and relies more on the supervisor.**

This text refers to the saved smoke file. If you load a different experiment, reassess the conclusion using its table.


## Tests

Run this cell to check the full suite, including Taxi regressions. Passing tests establishes implementation correctness, not better performance. Install the project requirements in the notebook's Python environment first.


In [3]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    cwd=PROJECT, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()


.....................                                                    [100%]
21 passed in 3.32s




## Run a new experiment (optional)

The notebook normally reads the saved results above. No additional experiment runs unless you set `RUN_EXPERIMENTS = True`. Choose a new output filename; existing results are protected.
After running, set `RESULT_FILE` above to the new path and rerun the main comparison.
The benchmark is a tabular DoorKey-inspired variant, not an exact MiniGrid reproduction.


In [4]:
from experiments.run_doorkey import run_sweep

RUN_EXPERIMENTS = False
NEW_OUTPUT = PROJECT / "results" / "doorkey_new_run.csv"

if RUN_EXPERIMENTS:
    run_sweep(
        NEW_OUTPUT,
        sizes=(5,),
        datasets=("noisy_expert",),
        n_trajs=(5,),
        seeds=tuple(range(3)),
        abstractions=("correct", "ignore_door"),
    )
    print(f"Saved {NEW_OUTPUT.name}; load it with RESULT_FILE above.")


## Additional diagnostics (hidden by default)

Enable only if needed: all agents and paired bootstrap intervals. Intervals resample paired seeds, not individual starting states. Three seeds are insufficient for a strong general conclusion.
Environment rules and implementation details are in [the report](../doorkey_implementation_report.md).


In [5]:
SHOW_DETAILS = False

if SHOW_DETAILS:
    from experiments.run_doorkey import paired_summary
    display(results.groupby(["env", "abstraction", "dataset", "n_traj", "agent"])[list(LABELS)]
            .mean().rename(columns=LABELS).round(2))
    display(paired_summary(results).round(3))
